# Faz 2 — Colab hatti

Yol haritasi Faz 2: **olcum aleti + baseline + butce**.

Kapi: *"Baseline sonucum AGAR makalesinin bildirdigi mertebede mi (dusuk cozunurlukte
mAP ~%59 civari bir bant), ve 61 kosu GPU butcesine sigiyor mu?"*

Sirasiyla:

1. GPU'yu dogrula
2. Kurulum
3. Repo + veri
4. **Olcum kodunun sanity testi** — bu gecmeden egitime baslama
5. Veri hattini kostur (convert + splits)
6. Duman testi (10 epoch)
7. G100 tam egitim, tek seed — **Faz 2'nin ana isi**
8. Degerlendirme (once val, sonra test)
9. Butce hesabi

> **Colab notu:** oturum kopabilir. G100 tam egitim saatler surerse
> `runs/` klasorunu Drive'a bagla (hucre 3) ve koptugunda
> `--resume` ile devam et. Aksi halde bastan baslarsin.

## 1. GPU

In [ ]:
!nvidia-smi
import torch, platform
print(f"\ntorch {torch.__version__} | CUDA {torch.version.cuda} | "
      f"kullanilabilir: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"kart: {p.name} | VRAM: {p.total_memory/1024**3:.1f} GB")
    # Butce hesabina bu kartin adini yazacaksin -- farkli kartlarda olculen
    # sureler karsilastirilamaz.


## 2. Kurulum

`pycocotools` opsiyonel ama **kur** — olcum kodunun capraz dogrulama testi ona
karsi kosuyor.

In [ ]:
# Surum SABIT: olcumler 8.4.118 ile yapildi. Colab her acilista
# guncelini ceker; farkli surum -> sureler karsilastirilamaz ve
# yolo26n.pt / cutmix durumu degisir (karar 3.46).
!pip install -q 'ultralytics==8.4.118' pycocotools wandb
import ultralytics; ultralytics.checks()


## 3. Repo ve veri

Iki secenek var. Birini sec, digerini yorumda birak.

**A) Drive** (onerilen — `runs/` Drive'da kalir, oturum kopunca kaybolmaz)
**B) GitHub** (repo private ise token gerekir)

In [ ]:
# --- A) Drive ---
from google.colab import drive
drive.mount('/content/drive')

PROJE = '/content/drive/MyDrive/agar-synth'   # repo buraya kopyalanmis olmali
%cd $PROJE
!ls -la

# --- B) GitHub (alternatif) ---
# !git clone https://<TOKEN>@github.com/<kullanici>/agar-synth.git /content/agar-synth
# %cd /content/agar-synth
# # veri Drive'dan gelsin:
# !ln -s /content/drive/MyDrive/AGAR /content/agar-synth/data/AGAR


## 4. Olcum kodunun sanity testi

**Bu gecmeden egitime baslama.** Kendi yazdigin metrige, dogru cevabini elle
bildigin vakalarla guvenmeden once inanma.

Beklenen: olcum `PASSED: 42   FAILED: 0` (pycocotools yoksa 36) ve
uretim `PASSED: 36   FAILED: 0`.

In [ ]:
!python src/eval/test_metrics.py
!python src/generate/test_generate.py

## 5. Veri hatti

Tam AGAR geldiyse burayi kostur. Demo pakette **val/test bos cikar** — bu
normal, 10 goruntu ve 7 tabaka ile bolme anlamli degil.

`make_splits.py` ciktisinda su uc satirin hepsi temiz olmali:
- `'/images/' -> '/labels/' ile etiket bulunamayan: 0`
- `train_10 ⊂ train_25 ⊂ train_50 ⊂ train : EVET`
- `train ∩ val / train ∩ test / val ∩ test : 0 goruntu`

In [ ]:
AGAR_KOK = 'data/AGAR'          # tam veri buraya  (demo icin: data/AGAR_representative)

!python src/convert.py --src $AGAR_KOK --out data/processed
!python src/make_splits.py --data data/processed --seed 42
!python src/analyze_manifest.py --data data/processed


In [ ]:
# Gozle dogrulama -- rastgele 30 goruntu, 5 sinifin hepsinden
!python src/check_labels.py --data data/processed --n 30

from IPython.display import Image as Im, display
import glob, random
for f in random.sample(sorted(glob.glob('data/processed/viz/*.jpg')), 3):
    print(f); display(Im(f, width=700))


## 6. Duman testi

Amac sonuc almak degil, **hattin calistigini** gormek: etiketler bulunuyor mu,
VRAM yetiyor mu, bir epoch ne kadar suruyor.

`Box(P R mAP50 ...)` satirinda mAP tamamen sifir kalirsa **etiketler
bulunamiyor** demektir — `dataset.yaml`'daki yollari kontrol et.

In [ ]:
!python scripts/train.py --config configs/base.yaml \
    --level 100 --seed 0 --epochs 10 --batch 4 \
    --name duman_testi --smoke


In [ ]:
# VRAM tepe degerine bak. Doluysa batch'i dusur, imgsz'yi DUSURME.
import json
m = json.load(open('runs/duman_testi/run_metrics.json'))
print(f"tepe VRAM     : {m['peak_vram_gb']} GB")
print(f"epoch basina  : {m['sec_per_epoch']:.1f} s   ({m['n_train']} goruntu)")
print(f"goruntu/saniye: {m['images_per_sec']:.1f}")


## 7. G100 tam egitim — Faz 2'nin ana isi

Tek seed. Amac hem **ust sinir** hem **sure olcumu**.

Colab'da saatler surebilir. Kopmaya karsi:
- `runs/` Drive'da olsun
- koparsa asagidaki resume hucresini kostur

In [ ]:
!python scripts/train.py --config configs/base.yaml \
    --level 100 --seed 0 --name G100_s0


In [ ]:
# Koptuysa devam et -- MUTLAKA train.py uzerinden (karar 3.42).
# Dogrudan YOLO(...).train(resume=True) cagirirsan run_metrics.json YAZILMAZ:
# sure, tepe VRAM ve git commit kaybolur. G100'un amaci "hem ust sinir HEM
# sure olcumu"ydu; o cagri olcum yarisini sessizce siler.
#
# !python scripts/train.py --config configs/base.yaml \
#     --level 100 --seed 0 --name G100_s0 --resume


## 8. Degerlendirme

**Sira onemli.** Once VAL — sayim conf esigi orada secilir. Sonra TEST, o esikle.

`evaluate.py --split test` esik verilmeden **calismaz**; test kumesinde esik
aramak sizintidir.

In [ ]:
!python src/eval/evaluate.py --weights runs/G100_s0/weights/best.pt \
    --data data/processed --split val --out runs/G100_s0/eval_val \
    --imgsz 1280 --tag G100_s0


In [ ]:
import json
ev = json.load(open('runs/G100_s0/eval_val/summary.json'))
CONF = ev['conf_thr_count']
print(f"VAL'da secilen conf esigi: {CONF}")
print("-> configs/base.yaml icindeki eval.conf_thr alanina YAZ ve commit et")


In [ ]:
!python src/eval/evaluate.py --weights runs/G100_s0/weights/best.pt \
    --data data/processed --split test --conf-thr $CONF \
    --out runs/G100_s0/eval_test --imgsz 1280 --tag G100_s0


In [ ]:
import json, pandas as pd
o = json.load(open('runs/G100_s0/eval_test/summary.json'))

print("=== FAZ 2 KAPISI, 1. YARISI ===")
print(f"mAP50-95 : {o['mAP50-95']:.4f}    <- AGAR makalesi dusuk cozunurlukte ~0.594")
print(f"mAP50    : {o['mAP50']:.4f}")
print(f"  small  : {o['mAP_small']:.4f}   <- kucuk koloniler, makalenin ana argumani")
print(f"  medium : {o['mAP_medium']:.4f}")
print(f"  large  : {o['mAP_large']:.4f}")
print(f"MAE      : {o['MAE']:.2f} koloni/goruntu     sMAPE: {o['sMAPE']:.1f}%")
print()
print(pd.read_csv('runs/G100_s0/eval_test/class_ap.csv', index_col=0).to_string())


## 9. Butce hesabi — kapinin ikinci yarisi

`--full-size`: tam AGAR'in lower-resolution + countable goruntu sayisi
(convert.py'nin "TUTULAN GORUNTU" satiri).

`--budget`: elindeki GPU-saat. Colab free'de gunluk birkac saat; Pro'da daha
fazla ama yine sinirli. **Gercekci bir sayi gir** — bu kapi ancak dogru
sayiyla is gorur.

In [ ]:
!python scripts/budget.py --metrics runs/G100_s0/run_metrics.json \
    --full-size 8000 --epochs 150 --budget 200 --scenarios \
    --gen-sec-per-image 8 --lora-hours 2 --lora-count 4


## 10. Kapi

| Soru | Cevap |
|---|---|
| Baseline AGAR mertebesinde mi? | mAP50-95 ~0.55–0.65 bandi ise **evet** |
| 61 kosu butceye sigiyor mu? | butce hesabi "SIGIYOR" diyorsa **evet** |

**Ikisi de evet →** Faz 3 (uretim hatti).

**Baseline dusukse** sirasiyla bak: etiketler bulunuyor mu (mAP 0'a yakinsa
kesin bu) → imgsz yeterli mi → epoch sayisi yetti mi (`patience` erken mi
kesti) → model boyutu (n → s).

**Butce sigmiyorsa** kisma sirasi: miktar taramasi seed 3→1, klasik kol 3→2,
ablasyon tek seviye, ikinci detektor tek konfigurasyon. **Ana gridi en sona
birak** — 8x5 makalenin belkemigi, n=3'e dusersen std savunman zayiflar.

Kapi gecince: `python scripts/collect.py` (sonuclari git'e al), `PROJE_DURUMU.md`'yi guncelle, `decisions.md`'ye Faz 2
kararlarini yaz, commit et.